# Terrain Feature Download + Sampling

This notebook downloads terrain rasters from **USGS 3DEP through Py3DEP**, samples the terrain values at each wildfire ignition point, engineers aspect-based features, and merges the terrain features into the current modeling table.


Core workflow:

```text
CAL FIRE + gridMET table
→ fire latitude/longitude points
→ AOI around fire points
→ download 3DEP DEM/slope/aspect rasters
→ sample raster values at each fire point
→ create northness/eastness from aspect
→ merge with modeling table
```

## 0. Package setup

Run this once in your environment if needed:

```powershell
pip install pandas numpy geopandas rasterio shapely pyproj py3dep rioxarray xarray
```

If Windows has trouble installing geospatial packages through pip, use conda/mamba:

```powershell
conda install -c conda-forge pandas numpy geopandas rasterio shapely pyproj py3dep rioxarray xarray
```


In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from shapely.geometry import box

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

TERRAIN_RAW_DIR = RAW_DIR / "terrain"
GEO_DIR = INTERIM_DIR / "geo"

for path in [TERRAIN_RAW_DIR, GEO_DIR, PROCESSED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

BASE_MODEL_PATH = PROCESSED_DIR / "calfire_with_gridmet.csv"
TERRAIN_FEATURES_PATH = PROCESSED_DIR / "terrain_features.csv"
OUTPUT_PATH = PROCESSED_DIR / "calfire_with_gridmet_terrain.csv"

print("Project root:", PROJECT_ROOT)
print("Base model:", BASE_MODEL_PATH)
print("Terrain raw folder:", TERRAIN_RAW_DIR)
print("Terrain features output:", TERRAIN_FEATURES_PATH)
print("Merged output:", OUTPUT_PATH)


Project root: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2
Base model: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\processed\calfire_with_gridmet.csv
Terrain raw folder: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\raw\terrain
Terrain features output: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\processed\terrain_features.csv
Merged output: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\processed\calfire_with_gridmet_terrain.csv


## 1. Load the current modeling table

This should be the final table from your gridMET pipeline.

Required columns:

- `gridmet_id`
- `Latitude`
- `Longitude`
- `AcresBurned`


In [2]:
df = pd.read_csv(BASE_MODEL_PATH)

required_cols = ["gridmet_id", "Latitude", "Longitude", "AcresBurned"]
missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["gridmet_id"] = df["gridmet_id"].astype(str)
df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")
df["AcresBurned"] = pd.to_numeric(df["AcresBurned"], errors="coerce")

pre_drop = len(df)
df = df.dropna(subset=["gridmet_id", "Latitude", "Longitude", "AcresBurned"]).copy()

# Keep California-ish coordinates.
df = df[df["Latitude"].between(32, 42)].copy()
df = df[df["Longitude"].between(-125, -113)].copy()

print("Rows loaded:", pre_drop)
print("Rows after coordinate/target filter:", len(df))
display(df[["gridmet_id", "Name", "Latitude", "Longitude", "AcresBurned"]].head())


Rows loaded: 2397
Rows after coordinate/target filter: 2397


,gridmet_id,Name,Latitude,Longitude,AcresBurned
0,fire_00000,Creek Fire,38.409580,-122.431720,65.0
1,fire_00001,Taglio Fire,37.217100,-121.080360,30.0
2,fire_00002,Tulloch Fire,37.927613,-120.528836,85.0
3,fire_00003,Metz Fire,36.381230,-121.200590,3876.0
4,fire_00004,Wheatland Fire,34.276000,-118.354000,156.0


## 2. Convert wildfire rows into geospatial points

Each wildfire row becomes a point geometry.

`EPSG:4326` means ordinary longitude/latitude coordinates.


In [3]:
fires_gdf = gpd.GeoDataFrame(
    df[["gridmet_id", "Name", "Latitude", "Longitude", "AcresBurned"]].copy(),
    geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),
    crs="EPSG:4326",
)

print("CRS:", fires_gdf.crs)
display(fires_gdf.head())


CRS: EPSG:4326


,gridmet_id,Name,Latitude,Longitude,AcresBurned,geometry
0,fire_00000,Creek Fire,38.409580,-122.431720,65.0,POINT (-122.43172 38.40958)
1,fire_00001,Taglio Fire,37.217100,-121.080360,30.0,POINT (-121.08036 37.2171)
2,fire_00002,Tulloch Fire,37.927613,-120.528836,85.0,POINT (-120.52884 37.92761)
3,fire_00003,Metz Fire,36.381230,-121.200590,3876.0,POINT (-121.20059 36.38123)
4,fire_00004,Wheatland Fire,34.276000,-118.354000,156.0,POINT (-118.354 34.276)


## 3. Create the terrain download AOI

This creates a bounding box around all wildfire points.

The AOI is saved as:

```text
data/interim/geo/fire_points_aoi.geojson
```

The AOI bounds are printed in:

```text
west south east north
```


In [4]:
buffer_degrees = 0.25

minx, miny, maxx, maxy = fires_gdf.total_bounds
aoi_bounds = (
    minx - buffer_degrees,
    miny - buffer_degrees,
    maxx + buffer_degrees,
    maxy + buffer_degrees,
)

aoi_wsen = f"{aoi_bounds[0]:.6f} {aoi_bounds[1]:.6f} {aoi_bounds[2]:.6f} {aoi_bounds[3]:.6f}"

aoi_gdf = gpd.GeoDataFrame(
    {"name": ["fire_points_aoi"]},
    geometry=[box(*aoi_bounds)],
    crs="EPSG:4326",
)

aoi_path = GEO_DIR / "fire_points_aoi.geojson"
aoi_gdf.to_file(aoi_path, driver="GeoJSON")

print("AOI west south east north:")
print(aoi_wsen)
print("Saved AOI:", aoi_path)
display(aoi_gdf)


AOI west south east north:
-124.612017 32.307546 -114.026308 42.244830
Saved AOI: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\interim\geo\fire_points_aoi.geojson


,name,geometry
0,fire_points_aoi,"POLYGON ((-114.02631 32.30755, -114.02631 42.2..."


## 4. Download terrain rasters from 3DEP

This cell uses `py3dep.get_map()` to download three terrain layers:

- `DEM`
- `Slope Degrees`
- `Aspect Degrees`

Default resolution:

```python
TERRAIN_RESOLUTION_M = 1000
```

That means each terrain pixel is about 1 km. This is coarse, but it is a good first pass for a statewide project and keeps the request size manageable.

If the download fails:
1. Try `TERRAIN_RESOLUTION_M = 2000`
2. Rerun the cell
3. Once everything works, optionally try 500 m or 250 m later


In [5]:
# If the request fails because the AOI is too large, increase this to 2000.
# If it works and you want more detailed terrain later, try 500 or 250.
TERRAIN_RESOLUTION_M = 1000

TERRAIN_LAYER_CONFIG = {
    "elevation": {
        "py3dep_layer": "DEM",
        "output_path": TERRAIN_RAW_DIR / "elevation.tif",
    },
    "slope_degrees": {
        "py3dep_layer": "Slope Degrees",
        "output_path": TERRAIN_RAW_DIR / "slope_degrees.tif",
    },
    "aspect_degrees": {
        "py3dep_layer": "Aspect Degrees",
        "output_path": TERRAIN_RAW_DIR / "aspect_degrees.tif",
    },
}

print("Terrain resolution:", TERRAIN_RESOLUTION_M, "meters")
print("AOI bounds:", aoi_bounds)
for name, cfg in TERRAIN_LAYER_CONFIG.items():
    status = "exists" if cfg["output_path"].exists() else "missing"
    print(f"{name:15s} {status:8s} {cfg['output_path']}")


Terrain resolution: 1000 meters
AOI bounds: (np.float64(-124.612017), np.float64(32.307546), np.float64(-114.026308), np.float64(42.24483))
elevation       missing  c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\raw\terrain\elevation.tif
slope_degrees   missing  c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\raw\terrain\slope_degrees.tif
aspect_degrees  missing  c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\raw\terrain\aspect_degrees.tif


In [6]:
def download_terrain_layer(layer_name, output_path, aoi_geometry, resolution_m=1000, overwrite=False):
    """Download one 3DEP terrain layer and save it as a GeoTIFF."""
    output_path = Path(output_path)

    if output_path.exists() and not overwrite:
        print(f"Skipping existing file: {output_path}")
        return output_path

    try:
        import py3dep
    except ImportError as exc:
        raise ImportError(
            "py3dep is not installed. Run: pip install py3dep rioxarray xarray"
        ) from exc

    print(f"Downloading {layer_name} at {resolution_m} m resolution...")
    print("This may take a while for a statewide AOI.")

    # CRS 5070 is recommended by current Py3DEP docs for requests.
    da = py3dep.get_map(
        layer_name,
        aoi_geometry,
        resolution=resolution_m,
        geo_crs="EPSG:4326",
        crs="EPSG:5070",
    )

    # Save using rioxarray's raster writer.
    output_path.parent.mkdir(parents=True, exist_ok=True)
    da.rio.to_raster(output_path)

    print("Saved:", output_path)
    return output_path


aoi_polygon = box(*aoi_bounds)

# Set overwrite=True only if you intentionally want to redownload.
OVERWRITE_TERRAIN = False

for feature_name, cfg in TERRAIN_LAYER_CONFIG.items():
    try:
        download_terrain_layer(
            layer_name=cfg["py3dep_layer"],
            output_path=cfg["output_path"],
            aoi_geometry=aoi_polygon,
            resolution_m=TERRAIN_RESOLUTION_M,
            overwrite=OVERWRITE_TERRAIN,
        )
        # Be polite to the remote service.
        time.sleep(2)
    except Exception as e:
        print(f"FAILED to download {feature_name}: {e}")
        print("Try increasing TERRAIN_RESOLUTION_M to 2000 and rerun this cell.")


This may take a while for a statewide AOI.
Saved: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\raw\terrain\elevation.tif
This may take a while for a statewide AOI.
Saved: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\raw\terrain\slope_degrees.tif
This may take a while for a statewide AOI.
Saved: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\raw\terrain\aspect_degrees.tif


## 5. Check that the rasters exist

You need these three files before sampling:

```text
data/raw/terrain/elevation.tif
data/raw/terrain/slope_degrees.tif
data/raw/terrain/aspect_degrees.tif
```


In [7]:
TERRAIN_RASTERS = {
    "elevation": TERRAIN_RAW_DIR / "elevation.tif",
    "slope_degrees": TERRAIN_RAW_DIR / "slope_degrees.tif",
    "aspect_degrees": TERRAIN_RAW_DIR / "aspect_degrees.tif",
}

for name, path in TERRAIN_RASTERS.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{name:15s} {status:8s} {path}")


elevation       FOUND    c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\raw\terrain\elevation.tif
slope_degrees   FOUND    c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\raw\terrain\slope_degrees.tif
aspect_degrees  FOUND    c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\raw\terrain\aspect_degrees.tif


## 6. Raster sampling function

This function samples a single-band raster at every wildfire point.

It automatically reprojects fire points into the raster's CRS before sampling.


In [8]:
def sample_single_band_raster(raster_path, points_gdf, output_col):
    raster_path = Path(raster_path)

    if not raster_path.exists():
        raise FileNotFoundError(f"Missing raster: {raster_path}")

    with rasterio.open(raster_path) as src:
        points_projected = points_gdf.to_crs(src.crs)
        coords = [(geom.x, geom.y) for geom in points_projected.geometry]

        values = [sample[0] for sample in src.sample(coords)]

        out = pd.DataFrame({
            "gridmet_id": points_gdf["gridmet_id"].astype(str).values,
            output_col: values,
        })

        out[output_col] = pd.to_numeric(out[output_col], errors="coerce")

        nodata_values = {-9999, -999, 32767, 65535}
        if src.nodata is not None:
            nodata_values.add(src.nodata)

        out.loc[out[output_col].isin(nodata_values), output_col] = np.nan

    return out

print("Raster sampling function ready.")


Raster sampling function ready.


## 7. Sample all available terrain rasters

This creates:

```text
gridmet_id | elevation | slope_degrees | aspect_degrees
```


In [9]:
sample_parts = []

for col_name, raster_path in TERRAIN_RASTERS.items():
    if not raster_path.exists():
        print(f"Skipping missing raster: {raster_path}")
        continue

    print(f"Sampling {col_name} from {raster_path.name}...")
    part = sample_single_band_raster(
        raster_path=raster_path,
        points_gdf=fires_gdf,
        output_col=col_name,
    )
    sample_parts.append(part)

if not sample_parts:
    raise FileNotFoundError(
        "No terrain rasters were found. Download the terrain rasters first, then rerun this cell."
    )

terrain = sample_parts[0]
for part in sample_parts[1:]:
    terrain = terrain.merge(part, on="gridmet_id", how="outer")

print("Terrain sample table:", terrain.shape)
display(terrain.head())


Sampling elevation from elevation.tif...
Sampling slope_degrees from slope_degrees.tif...
Sampling aspect_degrees from aspect_degrees.tif...
Terrain sample table: (2397, 4)


,gridmet_id,elevation,slope_degrees,aspect_degrees
0,fire_00000,223.804138,2.0,246.0
1,fire_00001,62.611248,1.0,80.0
2,fire_00002,254.468414,5.0,246.0
3,fire_00003,257.851990,11.0,134.0
4,fire_00004,395.886871,10.0,217.0


## 8. Engineer aspect features

Aspect is circular:

```text
0 degrees = north
360 degrees = north
```

So instead of modeling raw aspect alone, this creates:

```text
northness = cos(aspect)
eastness  = sin(aspect)
```


In [10]:
terrain_features = terrain.copy()

if "aspect_degrees" in terrain_features.columns:
    aspect = pd.to_numeric(terrain_features["aspect_degrees"], errors="coerce")

    # Some terrain sources use negative values for flat/no-aspect areas.
    aspect = aspect.where(aspect.between(0, 360), np.nan)

    aspect_rad = np.deg2rad(aspect)
    terrain_features["northness"] = np.cos(aspect_rad)
    terrain_features["eastness"] = np.sin(aspect_rad)

if "slope_degrees" in terrain_features.columns:
    terrain_features["slope_degrees"] = pd.to_numeric(terrain_features["slope_degrees"], errors="coerce")
    terrain_features.loc[terrain_features["slope_degrees"] < 0, "slope_degrees"] = np.nan

if "elevation" in terrain_features.columns:
    terrain_features["elevation"] = pd.to_numeric(terrain_features["elevation"], errors="coerce")

terrain_features.to_csv(TERRAIN_FEATURES_PATH, index=False)

print("Saved terrain features:", TERRAIN_FEATURES_PATH)
print("Terrain features shape:", terrain_features.shape)
display(terrain_features.head())


Saved terrain features: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\processed\terrain_features.csv
Terrain features shape: (2397, 6)


,gridmet_id,elevation,slope_degrees,aspect_degrees,northness,eastness
0,fire_00000,223.804138,2.0,246.0,-0.406737,-0.913545
1,fire_00001,62.611248,1.0,80.0,0.173648,0.984808
2,fire_00002,254.468414,5.0,246.0,-0.406737,-0.913545
3,fire_00003,257.851990,11.0,134.0,-0.694658,0.719340
4,fire_00004,395.886871,10.0,217.0,-0.798636,-0.601815


## 9. Merge terrain features with the modeling table

This creates:

```text
data/processed/calfire_with_gridmet_terrain.csv
```


In [11]:
base = pd.read_csv(BASE_MODEL_PATH)
terrain_features = pd.read_csv(TERRAIN_FEATURES_PATH)

base["gridmet_id"] = base["gridmet_id"].astype(str)
terrain_features["gridmet_id"] = terrain_features["gridmet_id"].astype(str)

merged = base.merge(terrain_features, on="gridmet_id", how="left")

merged.to_csv(OUTPUT_PATH, index=False)

print("Base table:", base.shape)
print("Terrain features:", terrain_features.shape)
print("Merged table:", merged.shape)
print("Saved:", OUTPUT_PATH)

terrain_cols = [c for c in ["elevation", "slope_degrees", "aspect_degrees", "northness", "eastness"] if c in merged.columns]
display(merged[["gridmet_id", "Name", "AcresBurned", "severity_tier"] + terrain_cols].head())


Base table: (2397, 174)
Terrain features: (2397, 6)
Merged table: (2397, 179)
Saved: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\processed\calfire_with_gridmet_terrain.csv


,gridmet_id,Name,AcresBurned,severity_tier,elevation,slope_degrees,aspect_degrees,northness,eastness
0,fire_00000,Creek Fire,65.0,Small,223.804140,2.0,246.0,-0.406737,-0.913545
1,fire_00001,Taglio Fire,30.0,Small,62.611248,1.0,80.0,0.173648,0.984808
2,fire_00002,Tulloch Fire,85.0,Small,254.468410,5.0,246.0,-0.406737,-0.913545
3,fire_00003,Metz Fire,3876.0,Large,257.852000,11.0,134.0,-0.694658,0.719340
4,fire_00004,Wheatland Fire,156.0,Medium,395.886870,10.0,217.0,-0.798636,-0.601815


## 10. Quality checks

Good signs:

- Missingness is low
- Elevation values look physically plausible
- Slope is usually between 0 and 90
- Northness/eastness are between -1 and 1


In [12]:
check = pd.read_csv(OUTPUT_PATH)

terrain_cols = [c for c in ["elevation", "slope_degrees", "aspect_degrees", "northness", "eastness"] if c in check.columns]

print("Terrain columns:", terrain_cols)

print("\nMissingness:")
display(check[terrain_cols].isna().mean().sort_values(ascending=False))

print("\nSummary statistics:")
display(check[terrain_cols].describe().T)

if "severity_tier" in check.columns:
    print("\nMedian terrain values by severity tier:")
    display(check.groupby("severity_tier")[terrain_cols].median().round(2))


Terrain columns: ['elevation', 'slope_degrees', 'aspect_degrees', 'northness', 'eastness']

Missingness:


elevation         0.0
slope_degrees     0.0
aspect_degrees    0.0
northness         0.0
eastness          0.0
dtype: float64


Summary statistics:


,count,mean,std,min,25%,50%,75%,max
elevation,2397.0,539.022458,497.040341,-62.598946,172.647960,402.984280,747.828550,3291.82
slope_degrees,2397.0,5.794326,5.617080,0.000000,2.000000,4.000000,8.000000,45.00
aspect_degrees,2397.0,187.091781,97.923448,0.000000,100.000000,206.000000,265.000000,359.00
northness,2397.0,-0.069789,0.669936,-1.000000,-0.707107,-0.139173,0.559193,1.00
eastness,2397.0,-0.131853,0.727556,-1.000000,-0.857167,-0.258819,0.601815,1.00



Median terrain values by severity tier:


,elevation,slope_degrees,aspect_degrees,northness,eastness
severity_tier,,,,,
Extreme,638.14,8.0,203.0,-0.22,-0.20
Large,571.13,5.0,197.0,-0.17,-0.18
Medium,393.13,4.0,205.0,-0.07,-0.22
Small,361.97,4.0,208.0,-0.14,-0.34


## 11. Final output

If this notebook works, your new terrain-enriched modeling table is:

```text
data/processed/calfire_with_gridmet_terrain.csv
```

Suggested next notebook:

```text
04_gridmet_terrain_eda.ipynb
```

Recommended README wording:

```text
Terrain features were sampled at each wildfire ignition coordinate using USGS 3DEP-derived elevation, slope, and aspect rasters. Aspect was converted into northness and eastness to avoid treating circular direction as a linear variable.
```
